<a href="https://colab.research.google.com/github/kanakesh2006/Langchain_for_Generative_AI_Concepts/blob/main/langchain_toolcalling/Tool_Calling_in_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [40]:
!pip install --upgrade -q langchain langchain-google-genai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.0/475.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.7/343.7 kB 27.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [4]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b thsi tool returns their product"""
  return a * b

In [5]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [6]:
multiply.name

'multiply'

In [7]:
multiply.description

'Given 2 numbers a and b thsi tool returns their product'

In [8]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [9]:
# tool binding

In [10]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [11]:
llm.invoke('hi')

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--ffa59122-c7ef-42e7-a5bc-dbf15118e972-0', usage_metadata={'input_tokens': 2, 'output_tokens': 33, 'total_tokens': 35, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 23}})

In [12]:
llm_with_tools = llm.bind_tools([multiply])

In [13]:
llm_with_tools.invoke("hi how are you")

AIMessage(content="I'm doing well, thank you! How can I help you today?", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--cc7ef592-9b5a-4d0d-9c2c-96e6e55efc0e-0', usage_metadata={'input_tokens': 59, 'output_tokens': 16, 'total_tokens': 75, 'input_token_details': {'cache_read': 0}})

In [14]:
query = HumanMessage('can you multiply 3 with 1000')

In [15]:
messages = [query]

In [16]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [17]:
result = llm_with_tools.invoke(messages)

In [18]:
messages.append(result)

In [19]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 3, "b": 1000}'}, '__gemini_function_call_thought_signatures__': {'40117de9-6671-4f12-9346-cc8c14d2d207': 'Co0CAXLI2nwdL8V/7JyArRKxC5WN64QV+kJFxttylVTnjZsc9XcZEsr/9DCdyOeO7aEdExArJNKqWigPXdVWsF1vblYLdGszZ4H2JQu6ds4uez4CP8L2myc190g180muXuLk3MHfm/ehyptPiDQeG0beFFumEm/cA/29SrKmi1fvtxf4y1x65+v0PZgTQJ9/L1fuZVZt652p3/Kp4yf8N4wIcovEge8K8psSAXDx54HvJWFu6y2bZFN09sHoNdlvi7zAp3VDs103OSlq+L9HY0kye6XrNomSf5M5osBcTVXbDO5qsNKJZNzC7vrt12XHQ5ZayqH/xqkTREOm8Lrsdgcycpq6EcX8dJw4gHXF2tE='}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--dca590bf-d8d4-4f26-8b75-d5d0172d0ef9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': '40117de

In [20]:
tool_result = multiply.invoke(result.tool_calls[0])

In [21]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='40117de9-6671-4f12-9346-cc8c14d2d207')

In [22]:
messages.append(tool_result)

In [23]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 3, "b": 1000}'}, '__gemini_function_call_thought_signatures__': {'40117de9-6671-4f12-9346-cc8c14d2d207': 'Co0CAXLI2nwdL8V/7JyArRKxC5WN64QV+kJFxttylVTnjZsc9XcZEsr/9DCdyOeO7aEdExArJNKqWigPXdVWsF1vblYLdGszZ4H2JQu6ds4uez4CP8L2myc190g180muXuLk3MHfm/ehyptPiDQeG0beFFumEm/cA/29SrKmi1fvtxf4y1x65+v0PZgTQJ9/L1fuZVZt652p3/Kp4yf8N4wIcovEge8K8psSAXDx54HvJWFu6y2bZFN09sHoNdlvi7zAp3VDs103OSlq+L9HY0kye6XrNomSf5M5osBcTVXbDO5qsNKJZNzC7vrt12XHQ5ZayqH/xqkTREOm8Lrsdgcycpq6EcX8dJw4gHXF2tE='}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--dca590bf-d8d4-4f26-8b75-d5d0172d0ef9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': '40117de

In [24]:
llm_with_tools.invoke(messages).content

'The product of 3 and 1000 is 3000.'

## Currency Converter Tool

In [25]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and taget currency
  """

  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()


@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [26]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [27]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1764892801,
 'time_last_update_utc': 'Fri, 05 Dec 2025 00:00:01 +0000',
 'time_next_update_unix': 1764979201,
 'time_next_update_utc': 'Sat, 06 Dec 2025 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 89.9074}

In [28]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [29]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [30]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [31]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [32]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [33]:
ai_message = llm_with_tools.invoke(messages)

In [34]:
messages.append(ai_message)

In [35]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'target_currency': 'USD', 'base_currency': 'INR'},
  'id': 'adff06a0-5b8d-4b67-a8d9-c1d2fa306e1e',
  'type': 'tool_call'}]

In [36]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [37]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "USD", "base_currency": "INR"}'}, '__gemini_function_call_thought_signatures__': {'adff06a0-5b8d-4b67-a8d9-c1d2fa306e1e': 'CrAKAXLI2nw+7DIS0CGaxYfGjtIT4+4jd3jhbR26tyrd6Oiwm9A9lYcemRudFGrIbr2tk2M+uRmDeHmXxucMPU/KJX/wGZDzukiQUCpiNmnWKW6+WgO4qytADx8+Grwe8enKBsGgULGHZXnaOO0Y8fvEV+RrO7epGeqq3hAnEcg2r1zD2+BSu64g3ypX4eeZEjn7NCSHiJHHsl8QjARoi4JZj8O//IyypZydOtqtiexq5z2wIzCsZnAo+mC1W3RABMZ13njTt6A+OSu+36nL3CD5RDYFTJu1T4JAr71za0OCj7uwzQMPg2LJ10xvShXIm9PucAfoxXDnfExp8fOv1JRla8DhiQsdw8A2qsUZjPwrVG/kNbbOsYNmQYgXuWvHnNWyUZOUfRXqlyr+1JmX1ENFbd4tbBac2tkVBMYBc/cVmSDuiYIM5qTGo27hEO4SjAV+w/B4DjhkCXv1zwTPPEFIgF1kJ4VqYAC3jSGxJqIifpZoYLpUO703mG0AL4lbkDPwWdSMem96z854JOkOY08M/XCPw6g0CGmQxFzPVQJuF7mziak5R/YkXX3KMtEazN0Oj

In [38]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.01112.\n\nI can provide you with the conversion factor. However, the `convert` function in the available tools is not designed to take the conversion factor directly or specify the target currency for the conversion. Therefore, I cannot directly convert 10 INR to USD using the available tools.'

In [39]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# Create a prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

# Step 5: Initialize the Agent ---
agent = create_tool_calling_agent(llm, [get_conversion_factor, convert], prompt)
agent_executor = AgentExecutor(agent=agent, tools=[get_conversion_factor, convert], verbose=True)


ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (/usr/local/lib/python3.12/dist-packages/langchain/agents/__init__.py)

In [ ]:
# --- Step 6: Run the Agent ---
user_query = "Hi how are you?"

response = agent_executor.invoke({"input": user_query})